In [1]:
import pandas as pd
from pathlib import Path

In [2]:
csv_file = next(
    Path("../../cleaned_data/road_accident_cleaned").glob("*.csv")
)

df = pd.read_csv(csv_file)

df.head()

,Accident_Index,Accident Date,Month,Day_of_Week,Year,Junction_Control,Junction_Detail,Accident_Severity,Latitude,Light_Conditions,...,Number_of_Casualties,Number_of_Vehicles,Police_Force,Road_Surface_Conditions,Road_Type,Speed_limit,Time,Urban_or_Rural_Area,Weather_Conditions,Vehicle_Type
0,200901BS70315,2021-06-16,Jun,Tuesday,2021,Give way or uncontrolled,T or staggered junction,Slight,51.486145,Daylight,...,1,1,Metropolitan Police,Dry,Single carriageway,30,2026-07-30T12:34:00.000+05:30,Urban,Fine no high winds,Car
1,200901BS70436,2021-07-08,Aug,Friday,2021,Give way or uncontrolled,T or staggered junction,Slight,51.492829,Daylight,...,1,1,Metropolitan Police,Dry,Single carriageway,30,2026-07-30T15:15:00.000+05:30,Urban,Fine no high winds,Car
2,200901CP00285,2021-10-30,Oct,Friday,2021,Give way or uncontrolled,T or staggered junction,Slight,51.510703,Daylight,...,1,1,City of London,Dry,Single carriageway,30,2026-07-30T08:50:00.000+05:30,Urban,Fine no high winds,Car
3,200901CW10181,2021-01-18,Jan,Sunday,2021,Auto traffic signal,Crossroads,Slight,51.492670,Daylight,...,1,2,Metropolitan Police,Wet or damp,Single carriageway,30,2026-07-30T13:05:00.000+05:30,Urban,Fine no high winds,Car
4,200901CW10194,2021-11-02,Feb,Wednesday,2021,Give way or uncontrolled,T or staggered junction,Slight,51.485691,Daylight,...,1,2,Metropolitan Police,Dry,Single carriageway,30,2026-07-30T06:36:00.000+05:30,Urban,Fine no high winds,Car


In [3]:
df["Accident_Severity"].value_counts()

Accident_Severity
Slight     256516
Serious     40083
Fatal        3892
Name: count, dtype: int64

In [4]:
severity_weight = {
    "Fatal": 5,
    "Serious": 3,
    "Slight": 1
}

df["Severity_Weight"] = df["Accident_Severity"].map(severity_weight)

In [5]:
risk_df = (
    df.groupby("Local_Authority_(District)")
      .agg(
          Total_Accidents=("Accident_Severity", "count"),
          Total_Casualties=("Number_of_Casualties", "sum"),
          Avg_Latitude=("Latitude", "mean"),
          Avg_Longitude=("Longitude", "mean"),
          Severity_Score=("Severity_Weight", "sum")
      )
      .reset_index()
)

risk_df.head()

,Local_Authority_(District),Total_Accidents,Total_Casualties,Avg_Latitude,Avg_Longitude,Severity_Score
0,Aberdeen City,434,488,57.153965,-2.132279,586
1,Aberdeenshire,672,887,57.291938,-2.351791,1116
2,Adur,279,364,50.837337,-0.282678,403
3,Allerdale,513,772,54.699908,-3.363552,675
4,Alnwick,20,31,55.349798,-1.753636,30


In [8]:
risk_df["Risk_Score"] = (
    risk_df["Severity_Score"] * 0.6 +
    risk_df["Total_Accidents"] * 0.3 +
    risk_df["Total_Casualties"] * 0.1
)

In [9]:
risk_df["Risk_Score"] = (
    risk_df["Risk_Score"] /
    risk_df["Risk_Score"].max()
) * 100

risk_df["Risk_Score"] = risk_df["Risk_Score"].round(2)

In [10]:
risk_df = risk_df.sort_values(
    "Risk_Score",
    ascending=False
)

risk_df.head(20)

,Local_Authority_(District),Total_Accidents,Total_Casualties,Avg_Latitude,Avg_Longitude,Severity_Score,Risk_Score
24,Birmingham,5948,8324,52.476970,-1.879527,7552,100.00
195,Leeds,4091,5759,53.804267,-1.544749,5297,69.69
404,Westminster,2801,3158,51.512098,-0.153708,4321,52.44
36,Bradford,2990,4413,53.812845,-1.790157,3820,50.79
208,Manchester,2869,4030,53.462190,-2.227518,3543,47.42
303,Sheffield,2684,3653,53.385904,-1.460074,3378,44.73
201,Liverpool,2536,3953,53.411394,-2.935710,3356,44.34
88,Cornwall,2546,3728,50.349246,-4.919553,3222,42.95
71,Cheshire East,2107,3146,53.192780,-2.339690,2973,38.20
90,County Durham,2213,3271,54.755442,-1.603927,2885,38.08


In [12]:
risk_df["Risk_Level"] = pd.cut(
    risk_df["Risk_Score"],
    bins=[0, 30, 60, 80, 100],
    labels=["Low", "Moderate", "High", "Very High"],
    include_lowest=True
)

risk_df.head()

,Local_Authority_(District),Total_Accidents,Total_Casualties,Avg_Latitude,Avg_Longitude,Severity_Score,Risk_Score,Risk_Level
24,Birmingham,5948,8324,52.476970,-1.879527,7552,100.00,Very High
195,Leeds,4091,5759,53.804267,-1.544749,5297,69.69,High
404,Westminster,2801,3158,51.512098,-0.153708,4321,52.44,Moderate
36,Bradford,2990,4413,53.812845,-1.790157,3820,50.79,Moderate
208,Manchester,2869,4030,53.462190,-2.227518,3543,47.42,Moderate


In [13]:
import os

os.makedirs("../data", exist_ok=True)

risk_df.to_csv(
    "../data/risk_zones.csv",
    index=False
)

print("Risk Zones Saved Successfully")

Risk Zones Saved Successfully
